# pandas 之三：Series 常用方法（速查手册）

---

这篇是**工具箱**，方法很多，不用背。做到三件事就够：

1. 知道**有这类方法**（遇到问题能想起来）
2. 会用 `s.<Tab>` 让编辑器补全
3. 记住最高频的那几个：`value_counts`、`describe`、`sort_values`、`isna`

下面按用途分成 8 组：

| 分组 | 代表方法 |
|---|---|
| 访问数据 | `head` `tail` `loc` `iloc` |
| 判断筛选 | `isin` `isna` `between` |
| 统计计算 | `sum` `mean` `std` `describe` `info` |
| 计数去重 | `count` `nunique` `value_counts` `drop_duplicates` |
| 排序取样 | `sort_values` `sort_index` `sample` `nlargest` |
| 数据修改 | `replace` `astype` `map` `apply` |
| 时序计算 | `shift` `diff` `pct_change` `rolling` |
| 分箱 | `cut` `qcut` |


In [ ]:
import pandas as pd
import numpy as np

s1 = pd.Series([1, 2, 3, 4, 5], index=['a', 'b', 'c', 'd', 'e'], name='数据')
print(s1)


---
## 一、访问数据

### 1. 按标签 / 位置取


In [ ]:
print("s1['c']     =", s1['c'], '    <- 标签取值')
print('s1.iloc[2]  =', s1.iloc[2], '    <- 位置取值（推荐）')


### 2. 布尔索引：给一个"真假列表"，取出 True 的那些

这是**最实用的筛选方式**，pandas 里天天用。


In [ ]:
mask = s1 > 2
print('掩码 mask =')
print(mask)
print('\n用掩码筛选：')
print(s1[mask])

# 也可以直接写条件
print('\n直接写条件 s1[s1 > 3]：')
print(s1[s1 > 3])


### 3. 切片访问

用**字符串标签**切片时，两端都包含（这是 pandas 的特殊规则）：
`s1['b':'d']` 会取出 b、c、d 三个。


In [ ]:
print("标签切片 s1['b':'d']（3 个）：")
print(s1['b':'d'])


### 4. 头尾访问：`head` / `tail`

数据量大的时候，**先看几行**是标准动作。


In [ ]:
print('head(2) 前两行：')
print(s1.head(2))

print('\ntail(2) 后两行：')
print(s1.tail(2))


---
## 二、判断与筛选


In [ ]:
# isin：值是否在给定集合里
s2 = pd.Series(['a', 'b', 'c', 'd', 'e'])
print('s2.isin([\'a\',\'c\',\'e\'])：')
print(s2.isin(['a', 'c', 'e']))
print('\n筛出在集合里的：', s2[s2.isin(['a', 'c', 'e'])].tolist())


In [ ]:
# isna / notna：是否是缺失值
s3 = pd.Series([1, 2, None, 4, np.nan])
print('s3 =')
print(s3)
print('\nisna（谁是缺失）：')
print(s3.isna())
print('\n取出缺失的：')
print(s3[s3.isna()])
print('\n非缺失的个数 =', s3.notna().sum())


> **`isna` 和 `isnull` 是同一个方法的两个名字**，效果完全一样。
> 记住一个就行（推荐 `isna`，更短）。
>
> 另外注意：`s3` 里混了 `None` 和 `np.nan`，
> pandas 会把 `None` 在数值列里也转成 `NaN`，**两者都算缺失**。


In [ ]:
# between：是否落在区间内（默认两端都包含）
s4 = pd.Series([1, 5, 10, 15, 20])
print('s4.between(5, 15)：')
print(s4.between(5, 15))
print('\n区间内的值：', s4[s4.between(5, 15)].tolist())


---
## 三、统计计算

Series 的统计方法**默认自动跳过 NaN**（这点和 Numpy 不同）。


In [ ]:
s5 = pd.Series([1, 2, 3, 4, 5])
print(f'sum   : {s5.sum()}')
print(f'mean  : {s5.mean()}')
print(f'min   : {s5.min()}')
print(f'max   : {s5.max()}')
print(f'std   : {s5.std():.10f}   <- 注意：pandas 默认 ddof=1（样本标准差）')
print(f'var   : {s5.var()}')


### ⚠️ pandas 和 Numpy 的标准差/方差 **不一样**！

这是一个**跨库对比时的大坑**：

| | 默认 `ddof` | 公式 |
|---|---|---|
| `np.std(a)` | **0** | 除以 n（总体） |
| `s.std()` | **1** | 除以 n-1（样本） |

同一组数据，两边算出来**必然不同**。


In [ ]:
a = np.array([1, 2, 3, 4, 5])
s = pd.Series([1, 2, 3, 4, 5])
print('np.std(a)          =', np.std(a), '  (ddof=0)')
print('s.std()            =', s.std(), '  (ddof=1)')
print('s.std(ddof=0)      =', s.std(ddof=0), '  <- 加上 ddof=0 就和 numpy 一致了')


### `describe()`：一行拿到全部描述性统计


In [ ]:
print(s5.describe())


输出里的名词：

| 名称 | 含义 |
|---|---|
| `count` | 非缺失值个数 |
| `mean` | 平均值 |
| `std` | 标准差（ddof=1） |
| `min` / `max` | 最小 / 最大值 |
| `25%` / `50%` / `75%` | 四分位数（50% 就是中位数） |


In [ ]:
# 如果有字符串列，describe 的侧重点会变
print('对象类型 Series 的 describe()：')
print(pd.Series(['cat', 'dog', 'cat', 'cat']).describe())


### `info()`：看"有没有缺失值"最快的办法


In [ ]:
s6 = pd.Series([1, 2, None, 4], name='样本')
s6.info()


**`Non-Null Count` 小于 `length` 就说明有缺失** —— 这是数据清洗的第一步。


---
## 四、计数与去重


In [ ]:
# count：非空值个数（注意不是"长度"）
s7 = pd.Series([1, 2, None, 4, 5])
print('len()   =', len(s7), '  <- 总共 5 个')
print('count() =', s7.count(), '  <- 非空只有 4 个')


In [ ]:
# value_counts：每个值出现多少次（数据分析里最常用的方法之一）
s8 = pd.Series(['a', 'b', 'a', 'c', 'b', 'a'])
print('value_counts：')
print(s8.value_counts())

print('\n加个 normalize=True 变成占比：')
print(s8.value_counts(normalize=True))


**`value_counts` 的价值**：
拿到一列数据，先看它"长什么样、有哪些取值、哪个最多"，
这是探索性数据分析（EDA）的第一步。它**默认按次数从多到少排序**。


In [ ]:
# nunique：唯一值有几个
s9 = pd.Series(['a', 'b', 'a', 'c', 'b', 'a'])
print('nunique =', s9.nunique(), '  <- 有 a,b,c 三种')


### ⚠️ 一个"隐形空格"的真实陷阱

下面这个 Series 里，第 5 个元素是 `'b '`（**带一个尾随空格**）。
它和 `'b'` 在 pandas 眼里**是两个不同的值**。


In [ ]:
s10 = pd.Series(['a', 'b', 'a', 'c', 'b ', 'a'])
print('原始值：', s10.tolist())
print('nunique =', s10.nunique(), '  <- 是 4，不是 3！因为 "b " ≠ "b"')
print('\n各值出现次数：')
print(s10.value_counts())

# 这类问题的处理方式：先 strip 去空格
print('\n去空格后 nunique =', s10.str.strip().nunique())


> **真实项目里最常见的脏数据就是"看不见的空格"**。
> 遇到"明明该去重却没去掉"的情况，第一反应就该是
> **用 `.str.strip()` 检查一遍**。


In [ ]:
# drop_duplicates：去重，保留第一次出现的
s11 = pd.Series(['a', 'b', 'a', 'c', 'b', 'a'])
print('去重结果：')
print(s11.drop_duplicates())


> **原稿错在哪**：原稿把这个例子的输出 dtype 写成了 `int64`。
> 但这是一个**字符串** Series，dtype 应该是 **`object`**（或 str）。
> 另外原稿保留了原索引（0,1,3），这是对的 —— `drop_duplicates`
> **默认不重置索引**，想要干净的 0,1,2 需要再 `.reset_index(drop=True)`。


In [ ]:
print('实际 dtype =', s11.drop_duplicates().dtype)
print('\n重置索引后：')
print(s11.drop_duplicates().reset_index(drop=True))


---
## 五、排序与取样


In [ ]:
s12 = pd.Series([3, 1, 4, 2, 5], index=['e', 'd', 'c', 'b', 'a'])
print('原 Series：')
print(s12)

# sort_index：按【索引】排序
print('\nsort_index（按 e,d,c,b,a 字母序）：')
print(s12.sort_index())

# sort_values：按【值】排序
print('\nsort_values（按值从小到大）：')
print(s12.sort_values())

# 降序
print('\nsort_values(ascending=False)：')
print(s12.sort_values(ascending=False))


**记法**：
- `sort_index` 排的是**左边那列**（索引）
- `sort_values` 排的是**右边那列**（数据）

> **原稿的小问题**：原稿的 `sort_index` 例子里，
> `s17 = pd.Series([3,1,4,2,5])` 用的是默认索引 `0,1,2,3,4`，
> 本身就已经是排好序的，所以 `sort_index()` 看不出变化，
> 起不到演示作用。这里换成了乱序索引，效果才明显。


In [ ]:
# nlargest / nsmallest：取前 N 大 / 前 N 小
s13 = pd.Series([88, 92, 79, 95, 85], index=['小明', '小红', '小刚', '小美', '小强'])
print('nlargest(2)：')
print(s13.nlargest(2))
print('\nnsmallest(2)：')
print(s13.nsmallest(2))


In [ ]:
# sample：随机抽样（加 random_state 让结果可复现）
s14 = pd.Series([1, 2, 3, 4, 5])
print('有放回抽 2 个：')
print(s14.sample(n=2, replace=True, random_state=0))
print('\n无放回抽 2 个：')
print(s14.sample(n=2, replace=False, random_state=0))


> **`random_state` 是什么？** 就是随机种子。
> 不设置的话，每次运行抽到的都不一样 ——
> 这在写教程、做实验复现时很麻烦。


---
## 六、数据修改


In [ ]:
# replace：替换值
s15 = pd.Series(['a', 'b', 'c', 'a', 'b'])
print('replace 之前：', s15.tolist())
print('replace 之后：', s15.replace({'a': 'x', 'b': 'y'}).tolist())
print('原 Series 变了吗？', s15.tolist(), '  <- 没变，replace 默认返回新对象')


In [ ]:
# astype：类型转换
s16 = pd.Series(['1', '2', '3'])
print('字符串转整数：', s16.astype(int).tolist(), ' dtype =', s16.astype(int).dtype)

# 转成 category（适合"只有几个固定取值"的列，省内存）
s17 = pd.Series(['男', '女', '男', '女'])
print('\n转 category：', s17.astype('category').tolist())
print('内存占用对比：')
print('  object   :', s17.memory_usage(), '字节')
print('  category :', s17.astype('category').memory_usage(), '字节')


In [ ]:
# map：按映射表逐个替换（相当于"查字典"）
s18 = pd.Series(['男', '女', '男'])
print('map 成布尔值：', s18.map({'男': True, '女': False}).tolist())

# apply：对每个元素应用一个函数（比 map 更自由）
s19 = pd.Series([1, 2, 3, 4])
print('apply 平方：', s19.apply(lambda x: x ** 2).tolist())


**`map` 和 `apply` 怎么选？**

| | 适用场景 |
|---|---|
| `map` | 简单的"值 -> 值"映射（字典 / 函数） |
| `apply` | 需要更复杂逻辑、能拿到元素做任意计算 |

一般来说，**能用 `map` 就用 `map`**（更快）。


---
## 七、时序计算（`shift` / `diff` / `pct_change` / `rolling`）

这几个在**时间序列分析**里是核心，也常用于"和上一期比"的场景。


In [ ]:
s20 = pd.Series([100, 110, 121, 133, 146], name='销量')
print('原始：')
print(s20)

# shift：整体向下移动 n 位，空出来的位置填 NaN
print('\nshift(1)：')
print(s20.shift(1))

# diff：和后一个的差（等价于 s - s.shift(1)）
print('\ndiff(1)：')
print(s20.diff(1))

# pct_change：环比增长率
print('\npct_change()（环比增长率）：')
print(s20.pct_change())


验证一下 `diff` 和 `pct_change` 的关系：

```
diff(1)       = 本期 - 上期
pct_change()  = (本期 - 上期) / 上期
```


In [ ]:
print('验证：s20 - s20.shift(1) 等于 diff(1) 吗？')
print((s20 - s20.shift(1)).equals(s20.diff(1)))


### `rolling`：移动窗口计算（比如"最近 3 天的平均"）


In [ ]:
print('rolling(3).mean() 最近 3 期平均：')
print(s20.rolling(window=3).mean())

print('\n加 min_periods=1，让开头也出数：')
print(s20.rolling(window=3, min_periods=1).mean())


**理解 `rolling(3).mean()`**：第 3 个数 = (第1个+第2个+第3个)/3，以此类推。
所以前两个位置是 `NaN`（凑不够 3 个数）。


---
## 八、分箱（`cut` / `qcut`）

**分箱 = 把连续的数值切成几个区间**，常用于"年龄分段""成绩分档"。


In [ ]:
# cut：按【等宽】分箱（每个区间的数值跨度一样）
scores = pd.Series([55, 62, 71, 78, 85, 92, 96, 100])
print('等宽分 3 箱：')
print(pd.cut(scores, bins=3))


In [ ]:
# 指定边界 + 标签（更常用）
bins = [0, 60, 80, 100]
labels = ['不及格', '及格', '优秀']
graded = pd.cut(scores, bins=bins, labels=labels, right=False)
print('指定边界分箱：')
print(graded)
print('\n各档人数：')
print(graded.value_counts())


> **`right=False` 是什么意思？**
> 默认 `right=True` 表示区间是"左开右闭" `(0, 60]`；
> 设成 `False` 就变成"左闭右开" `[0, 60)`。
>
> **为什么常用 `right=False`？** 因为要表达"60 分算及格"时，
> `[60, 80)` 才符合直觉（60 落在"及格"里）。
> 用默认的 `(0,60]` 会把 60 分归到"不及格"，容易出事故。


In [ ]:
# qcut：按【等频】分箱（每个区间的元素个数差不多）
print('等频分 4 箱（每箱人数尽量相等）：')
q = pd.qcut(scores, q=4)
print(q)
print('\n各档人数：')
print(q.value_counts().sort_index())


### `cut` vs `qcut` —— 一张表说清

| | 分箱依据 | 特点 | 什么时候用 |
|---|---|---|---|
| `pd.cut` | **值域等分** | 每箱跨度一样，但人数可能不均 | 有明确的业务阈值（如 60/80 分） |
| `pd.qcut` | **数量等分** | 每箱人数一样，但跨度不均 | 想让每档人数均匀（如四分位） |


---
## 九、综合小练习（原稿的 3 个案例）

### 案例 1：学生数学成绩


In [ ]:
np.random.seed(0)
math_scores = pd.Series(np.random.randint(50, 101, size=10),
                        index=[f'Student_{i + 1}' for i in range(10)],
                        name='数学成绩')
print('数学成绩：')
print(math_scores)

print('\n平均分：', round(math_scores.mean(), 2))
print('最高分：', math_scores.max())
print('最低分：', math_scores.min())

print('\n高于平均分的学生：')
print(math_scores[math_scores > math_scores.mean()])
print('人数：', math_scores[math_scores > math_scores.mean()].count())


### 案例 2：一周最高温度


In [ ]:
np.random.seed(0)
temperatures = pd.Series(
    np.random.randint(20, 40, size=7),
    index=['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'],
    name='最高温度',
)
print('一周温度：')
print(temperatures)

print('\n超过 30 度的天数：')
print(temperatures[temperatures > 30])

print('\n平均温度：', round(temperatures.mean(), 2))

print('\n从高到低排序：')
print(temperatures.sort_values(ascending=False))


**温度变化最大的两天** —— 这题的关键是 `diff()`：

```
diff()          -> 相邻两天的温差（今天的 - 昨天的）
.abs()          -> 取绝对值（降温也要算"变化大"）
.sort_values()  -> 差异从大到小
.head(2)        -> 取最大的两个
```


In [ ]:
changes = temperatures.diff().abs().sort_values(ascending=False)
print('温差变化最大的两天（显示的是"后一天"的日期）：')
print(changes.head(2))


### 案例 3：股票收盘价


In [ ]:
np.random.seed(0)
closing = pd.Series(
    np.random.uniform(100, 200, size=10),
    index=[f'2023-01-{i:02d}' for i in range(1, 11)],
    name='收盘价',
)
print('收盘价：')
print(closing.round(2))

# 每日收益率 = (今天 - 昨天) / 昨天 = pct_change()
returns = closing.pct_change() * 100
print('\n每日收益率（%）：')
print(returns.round(2))

print('\n收益率最高的日期：', returns.idxmax())
print('收益率最低的日期：', returns.idxmin())
print('波动率（收益率标准差）：', round(returns.std(), 2))


> **`idxmax()` 和 `max()` 的区别**（和 Numpy 的 `argmax` 对照记）：
> - `max()` 给你**最大的值**
> - `idxmax()` 给你**最大值的索引**（"哪一天"）
>
> 在 Series 里 `idxmax` 比 Numpy 的 `argmax` 更好用 ——
> 它直接返回**标签**（日期），而不是位置数字。


In [ ]:
print('对照一下：')
print('  max()    =', round(returns.max(), 2), '（收益率是多少）')
print('  idxmax() =', returns.idxmax(), '（是哪一天）')


---
## 十、常见坑清单

| 坑 | 现象 | 正解 |
|---|---|---|
| `s.std()` 和 `np.std()` 结果不同 | 数字对不上 | ddof 默认不同（1 vs 0） |
| `s.count()` 以为等于 `len(s)` | 数字偏小 | `count` 只数**非空值** |
| `'b '` 和 `'b'` 当成一样 | 去重没去干净 | 用 `.str.strip()` |
| `drop_duplicates()` 后索引是跳的 | 索引不连续 | 加 `.reset_index(drop=True)` |
| `sort_index` / `sort_values` 用混 | 排的不是想排的那列 | index 排索引，values 排数据 |
| `pct_change()` 第一个值是 NaN | 以为出错 | 第一个没有"上一期"，正常 |
| `rolling(3)` 前两个是 NaN | 以为出错 | 凑不满 3 个数，正常 |
| `pd.cut` 边界归属不清 | 60 分被判成不及格 | 用 `right=False` 明确区间 |
| `cut` / `qcut` 用混 | 人数或跨度不均 | 等宽用 cut，等频用 qcut |
| `replace`/`astype` 以为改了原对象 | 原 Series 没变 | 它们**返回新对象**，要接收返回值 |

---

## 十一、动手练习

**练习 1**：创建 `pd.Series([12, 7, 15, 9, 20, 7, 13], name='销量')`，算出总和、平均、标准差，
并说明 `std()` 用的是 ddof=1 还是 0。

**练习 2**：有一列城市名 `['北京','上海','北京','广州','上海','北京']`，
统计每个城市出现的次数和占比。

**练习 3**：给定 `s = pd.Series([100, 105, 99, 110, 120])`，
计算环比增长率，并找出增长率最高的那一期。

**练习 4**：有一组年龄 `[12, 25, 38, 45, 67, 19, 55, 8]`，
用 `pd.cut` 分成 `少年/青年/中年/老年` 四档并统计各档人数。

**练习 5**：用 `pd.qcut` 把上面的年龄分成 4 等份，比较和 `cut` 的差别。

**练习 6**：找出这组数据里"和上一期差值最大"的位置：
`s = pd.Series([10, 30, 25, 80, 20])`。

---

**答案思路（先自己做）**

```python
# 1
s = pd.Series([12, 7, 15, 9, 20, 7, 13], name='销量')
print(s.sum(), s.mean(), s.std())
print(s.std(ddof=0))     # 默认是 ddof=1（样本标准差）

# 2
cities = pd.Series(['北京', '上海', '北京', '广州', '上海', '北京'])
print(cities.value_counts())
print(cities.value_counts(normalize=True))

# 3
s = pd.Series([100, 105, 99, 110, 120])
r = s.pct_change()
print(r)
print(r.idxmax())

# 4
age = pd.Series([12, 25, 38, 45, 67, 19, 55, 8])
groups = pd.cut(age, bins=[0, 18, 35, 55, 100],
                labels=['少年', '青年', '中年', '老年'], right=False)
print(groups.value_counts().sort_index())

# 5
print(pd.qcut(age, 4).value_counts().sort_index())

# 6
s = pd.Series([10, 30, 25, 80, 20])
print(s.diff().abs().idxmax())
```
